# P4-A2 — Human-in-the-Loop (approve before acting)

Back in P2-A6 you flagged the key agent safeguard: *pause for confirmation before a destructive action.* Now you build it. LangGraph's `interrupt_before` stops the graph **right before the tools node** — the agent has *decided* to call a tool but hasn't run it yet. You inspect the pending action, then **approve** (resume) or **reject** (don't).

This is built directly on P4-A1: the interrupt only works because the checkpointer can freeze the state mid-run and resume it later.

In [1]:
# --- Setup (given) ---
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')

ORDERS = {'A123': {'status': 'shipped'}, 'B456': {'status': 'processing'}}

@tool
def get_order_status(order_id: str) -> dict:
    """Get the status of an order by ID (safe, read-only)."""
    return ORDERS.get(order_id, {'status': 'not_found'})

@tool
def cancel_order(order_id: str) -> dict:
    """Cancel an order by ID (DESTRUCTIVE — changes state)."""
    if order_id in ORDERS:
        ORDERS[order_id]['status'] = 'cancelled'
        return {'order_id': order_id, 'result': 'cancelled'}
    return {'order_id': order_id, 'result': 'not_found'}

TOOLS = [get_order_status, cancel_order]

def build_agent():
    """Agent that PAUSES before running any tool (interrupt_before=['tools'])."""
    bound = llm.bind_tools(TOOLS)
    def agent_node(state):
        return {'messages': [bound.invoke(state['messages'])]}
    g = StateGraph(MessagesState)
    g.add_node('agent', agent_node)
    g.add_node('tools', ToolNode(TOOLS))
    g.add_edge(START, 'agent')
    g.add_conditional_edges('agent', tools_condition)
    g.add_edge('tools', 'agent')
    return g.compile(checkpointer=InMemorySaver(), interrupt_before=['tools'])

def pending_tool_calls(agent, config):
    """Helper: if the agent is paused before tools, return the tool calls it WANTS to run."""
    state = agent.get_state(config)
    if not state.next:                      # state.next is empty if not interrupted
        return None
    return state.values['messages'][-1].tool_calls

print('ready')

ready


## Task 1 — Run, and watch it PAUSE before acting
Invoke the agent with *"Please cancel order B456."* and a `thread_id` config. Because of `interrupt_before=['tools']`, it stops before executing. Use `pending_tool_calls(agent, config)` to print the tool call it's waiting to run, and confirm `ORDERS['B456']` is **still 'processing'** (nothing happened yet).
<br>Hint: `agent = build_agent()`; `config = {'configurable': {'thread_id': 't1'}}`; `agent.invoke({'messages':[{'role':'user','content':'...'}]}, config=config)`.

In [2]:
# TODO: invoke the cancel request; show the pending tool call + that ORDERS is unchanged

agent = build_agent()
config = {'configurable': {'thread_id': 't1'}}

agent.invoke({'messages': [{'role': 'user', 'content': 'Please cancel order B456.'}]}, config=config)

pending = pending_tool_calls(agent, config)
print('Pending tool calls (decided, not yet run):')
for tc in pending:
    print(f"  -> {tc['name']}({tc['args']})")

print("\nORDERS['B456'] is still:", ORDERS['B456']['status'])   # 'processing' — nothing happened yet


Pending tool calls (decided, not yet run):
  -> cancel_order({'order_id': 'B456'})

ORDERS['B456'] is still: processing


## Task 2 — APPROVE: resume and let it run
The human approves. Resume the graph by invoking with `None` as the input and the **same config** — LangGraph continues from where it paused, runs the tool, and finishes. Print the final answer and confirm `ORDERS['B456']` is now `'cancelled'`.
<br>Hint: `agent.invoke(None, config=config)` resumes a paused graph.

In [3]:
# TODO: approve -> resume with invoke(None, config) -> verify the order is cancelled

result = agent.invoke(None, config=config)   # None = "no new input, just continue from the pause"

print('Final answer:', result['messages'][-1].content)
print("ORDERS['B456'] is now:", ORDERS['B456']['status'])   # 'cancelled'


Final answer: Order B456 has been successfully cancelled.
ORDERS['B456'] is now: cancelled


## Task 3 — REJECT: decline and don't run it
Now the safety case. On a **fresh thread_id**, ask to cancel `A123`. It pauses again. This time the human says **no** — so you simply *don't* resume. Show that `ORDERS['A123']` is **still 'shipped'** (the destructive action never executed).
<br>Goal: the agent proposed an action; the human had the final say and blocked it. That's the whole point of HITL.
<br>(Bonus: print a message back to the 'user' explaining the action was declined.)

In [4]:
# TODO: fresh thread; pause on cancel A123; DON'T resume; show A123 unchanged

config2 = {'configurable': {'thread_id': 't2'}}
agent.invoke({'messages': [{'role': 'user', 'content': 'Please cancel order A123.'}]}, config=config2)

pending = pending_tool_calls(agent, config2)
print('Agent wants to run:', [(tc['name'], tc['args']) for tc in pending])

# The human says NO -> we simply never call invoke(None, config2).
print("\nAction DECLINED by human. Not resuming.")
print("ORDERS['A123'] is still:", ORDERS['A123']['status'])   # 'shipped' — never executed

# Bonus: message back to the user
print("\nTo user: I did not cancel order A123 because the request was declined. "
      "Let me know if you'd like to proceed.")


Agent wants to run: [('cancel_order', {'order_id': 'A123'})]

Action DECLINED by human. Not resuming.
ORDERS['A123'] is still: shipped

To user: I did not cancel order A123 because the request was declined. Let me know if you'd like to proceed.


## Task 4 — Explain (in your own words)
1. Why is human-in-the-loop important specifically for *agents* (vs. a plain chatbot)? What class of tool should always gate behind it?
2. How does the interrupt/resume mechanism rely on what you built in P4-A1? (What makes 'pause now, resume later' even possible?)

> _your answer here_

. A plain chatbot only produces text — the worst case is a wrong or awkward sentence, which a human reads before doing anything. An agent takes real actions in the world through tools: it cancels orders, sends emails, moves money, deletes records. Those actions are often irreversible and execute without a human in the path, so a single bad decision (a hallucinated order ID, a misread request) causes real damage that can't be undone by "ignoring the reply." Human-in-the-loop reinserts a person at exactly the dangerous moment. The class of tool that should always gate behind approval is anything destructive, irreversible, or externally consequential — state mutations (cancel/delete/refund), spending money, sending communications, anything touching production systems. Read-only tools like get_order_status are safe to run freely; it's the writes that need a gate.
2. Interrupt/resume is only possible because of the checkpointer from P4-A1. To "pause now, resume later," the graph must be able to freeze its entire state mid-run and reload it on a later call — and that frozen snapshot is precisely what the checkpointer saves after each step, keyed by thread_id. So interrupt_before=['tools'] works like this: the agent node runs and decides on a tool call, the checkpointer saves that state, and the graph stops with state.next pointing at the (not-yet-run) tools node. The pending action lives in storage, not in a live function call. Approving is just invoke(None, config) with the same thread_id — LangGraph reloads that exact frozen state and continues; rejecting is simply never resuming that thread, so the saved-but-unexecuted tool call just sits there and the world is never changed. No checkpointer → no saved mid-run state → nothing to pause or resume.